## Splash Test 💧💧💧

<div class="alert alert-block alert-success">

## Part 3: Executable Code & Running in Terminal

As mentioned in Part 2, we will go over the code in `shallow_splash.py` in detail here, but then give instructions for how to run in terminal. 

This is because the Cubed-Sphere format requires at least 6 CPU cores to run, which is not possible to achieve in a single-core Jupyter notebook. 

</div>

<div class="alert alert-warning">

## $\texttt{PADDLE}$ Splash Executable Code 


<div class="alert alert-info">

## First, import the necessary packages. 

Here, there are few key differences to things we imported in previous tutorials. 

The first is that for shallow splash **density = geopotential**. This is because density is assumed constant in the shallow splash, so we assign geopotential to the density slot in the tensor. 

This is also the first time we import the keys for the velocity components (kIV2 and kIV3), which we use to explicitly set the initial condition where velocities are 0. The other primitive variables (kIV1) is always 0 for a single layer in the vertical direction.

We also import snapy functions that are essential to the Cubed-Sphere geometry, which we will describe more in detail below. 

```python
# Torch is the basic computational backend, equivalent in some ways to numpy but can work on CPU and GPU
import torch

# Import numpy in order to get pi 
import numpy as np

# os is used to create the output directory for outputs 
import os 

# snapy is a package developed by the same developers as PADDLE and CANOE that performs dynamical calculations 
# see https://github.com/chengcli/snapy

# Here we import the main module (MeshBlock) and options (MeshBlockOptions) used in all PADDLE simulations
# and the named locations of geopotential (where, for shallow splash the geopotential lives in the density slot of the tensor)
# and the horizontal velocities (kIV2 and kIV3)
from snapy import MeshBlockOptions, MeshBlock
from snapy import kIDN as kIGP
from snapy import kIV2, kIV3

# We also import modules needed for the cubed-sphere geometry, described in detail below. 
from snapy.distributed import get_rank, get_layout
from snapy.coord import get_cs_face_name, cs_ab_to_lonlat


<div class="alert alert-info">

## Custom user output variables 

Here, we opt to not define at custom outputs (like temperature) since we are only interested in how the geopotential evolves over the sphere. (which is already included in the primitive variable outputs)

<div class="alert alert-info">

## Generate the $\texttt{MeshBlock}$ Object

```python
# set hydrodynamic options from our yaml file
op = MeshBlockOptions.from_yaml("shallow_splash.yaml", verbose=False)

# Set output directory 
output_directory = "./splash_results"
os.makedirs(output_directory, exist_ok = True)
op.output_dir(output_directory)

# initialize block 
block = MeshBlock(op)

<div class="alert alert-info">

# Let torch know if you are using CPU or GPU

```python
# use cuda if available
if torch.cuda.is_available() and op.layout().backend() == "ucx":
    device = torch.device(block.device())
else:
    device = torch.device("cpu")

block.to(device)

<div class="alert alert-info">

# Set up the Cubed-Sphere Coordinate System

Here, we follow the same flow as what was done in *Straka* and *Robert*, but utilize helper functions to pull out a latitude-longitude grid for the cubed-sphere in order to make generating the initial condition more intuitive. 

`Need details here`

```python
# get handles to specific modules

# Coordinates of simulation domain stored in MeshBlock
# We will need this to generate the meshgrid we will define our initial condition of the simulation on top of 
coord = block.module("coord")

# For Cubed Sphere, need to first set up the meshgrid of the simulation 
r = get_rank()
layout = get_layout()
rx, ry, face_id = layout.loc_of(r)
face = get_cs_face_name(face_id)

# coord.buffer("x2v") 
# returns a 1D tensor with length = number of cells in x2 direction + 2 x ghost zones per face of the cube
# As defined in the .yaml file, this would be 48 cells + 2 x 3 ghost zones = 54 
beta, alpha, r_planet = torch.meshgrid(
    coord.buffer("x3v"), coord.buffer("x2v"), coord.buffer("x1v"), indexing="ij"
)

# Helper function that turns the two anglular and one radius dimension into latitude, longitude 
lon, lat = cs_ab_to_lonlat(face, alpha, beta)

# The number of points in each dimension
# This is identical to the number of cells 
nc3 = coord.buffer("x3v").shape[0]
nc2 = coord.buffer("x2v").shape[0]
nc1 = coord.buffer("x1v").shape[0]

<div class="alert alert-info">

## Creating the Initial Condition

Now, we want to actually define our simulation's initial condition. 

In order to do this, we follow a similar flow as we did in **Part 1**, where we create our 3-dimensional grid and then define the primitive variables at each point in the grid. 

`Need more details here, why isn't pressure a variable`
<div>

```python
# Hardcode the number of variables (4 = number of primitive variables in shallow splash -> geopotential, 3 velocities)
nvar = 4

# Define a 4D tensor storing primitive variables at each simulation coordinate
# In athena++, w signifies the primitive variables 
w = torch.zeros((nvar, nc3, nc2, nc1), device=device)

# As defined in Part 1
# Distance from every point on the lat-lon grid from the North Pole 
arc_length = r_planet * ((np.pi / 2.0) - lat)

# Create geo-potential initial condition 
# When arc length is less than R_perturbation it is phi_0 + dphi, 
# Otherwise, it is just phi_0
# Here we also ensure we are only looking at positive latitudes (>pi/4 = 45 Deg N)
w[kIGP] = Phi_0
w[kIGP][torch.logical_and(arc_length <  R_perturbation, lat > np.pi / 4.0)] += dPhi

# We also set the horizontal/angular velocities to be 0 
w[kIV2] = 0.0
w[kIV3] = 0.0

`Set the initial condition into block`

```python
# Create an empty block variables dictionary
block_vars = {}

# Populate the primitive variables with the initial condition we created above
block_vars["hydro_w"] = w

# Initialize the MeshBlock (block) object with the block variables 
block_vars, current_time = block.initialize(block_vars)

<div class="alert alert-info">

## Starting the Simulation

Below, we actually start integrating the model.

The code below will be more-or-less the same for all $\texttt{PADDLE}$ python scripts, and can be taken as-is for now. 

<div>

```python

# Initialize the output before the simulation starts, compute until the desired time 
block.make_outputs(block_vars, current_time)

# Run the simulation in a loop
while not block.intg.stop(block.inc_cycle(), current_time):

    # Each time step of the simulation is determined by the cfl number and the sound speed 
    dt = block.max_time_step(block_vars)

    # Output to let us know the code is running
    block.print_cycle_info(block_vars, current_time, dt)

    # For each cycle, a multi-stage method (here rk3) is used to advance block_vars
    for stage in range(len(block.intg.stages)):
        block.forward(block_vars, dt, stage)

    # Check for any errors
    err = block.check_redo(block_vars)
    if err > 0:
        continue  # redo current step
    if err < 0:
        break  # terminate

    # Progress the time and make outputs 
    current_time += dt
    block.make_outputs(block_vars, current_time)

# Make the final outputs and clean up the internal states in MeshBlock
block.finalize(block_vars, current_time)

<div class="alert alert-info">

## Step-by-step running Splash in terminal


1. Open a terminal and activate your $\texttt{PADDLE}$ conda environment
2. `cd` into the directory storing the `shallow_splash.yaml` and `shallow_splash.py` (the directory with this notebook in it)
3. Run the following code in terminal (for cubed-sphere geometries, 6 = number of cores = 6 * nb2^2)
    ```
    torchrun --nproc-per-node=6 shallow_splash.py
    ```

`Why is it out0 instead of 1?` 
</div>

<div class="alert alert-block alert-success">

Now we have our `main.nc` file with our results. 

On we go to **Part 4**, where we view and analyze the results in a notebook. 